In [6]:
# !pip install youtube-transcript-api
#!pip install chromadb
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 10.9 MB/s  0:00:00 eta 0:00:01

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [7]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores import Chroma
#from langchain_core.schema import Document
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
import os

In [8]:
load_dotenv()

True

### Document Loader

In [10]:
document = YouTubeTranscriptApi()
video_id = 'HC5Lquu7V0A'
transcript = document.fetch(video_id)

In [11]:
transcript

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='hello everyone in my previous video I', start=0.04, duration=3.919), FetchedTranscriptSnippet(text='have already shown you how you can do', start=1.959, duration=4.721), FetchedTranscriptSnippet(text='text summarization using Haring library', start=3.959, duration=5.001), FetchedTranscriptSnippet(text='in this video I will show you how you', start=6.68, duration=5.28), FetchedTranscriptSnippet(text='can do text summarization from a PDF it', start=8.96, duration=5.12), FetchedTranscriptSnippet(text='means that you have a text in a PDF and', start=11.96, duration=4.48), FetchedTranscriptSnippet(text='you want to summarize the content using', start=14.08, duration=5.44), FetchedTranscriptSnippet(text='that PDF using Haring liary before', start=16.44, duration=4.2), FetchedTranscriptSnippet(text='starting let me give a short', start=19.52, duration=4.079), FetchedTranscriptSnippet(text='introduction about myself welcome to ad', sta

In [12]:
text = " ".join(doc.text for doc in transcript)

In [13]:
text

"hello everyone in my previous video I have already shown you how you can do text summarization using Haring library in this video I will show you how you can do text summarization from a PDF it means that you have a text in a PDF and you want to summarize the content using that PDF using Haring liary before starting let me give a short introduction about myself welcome to ad Academy the main Moto of this channel is AI for arm jenta my name is Dr Ayan deat I am an IIT Delhi alumni and FB research scholar at Harvard University I have total 9 plus years of experience in the field of artificial intelligence deep learning machine learning NLP generative AI let's watch this video Welcome to My meta ver you can do teex separation in just three simple steps in the first step you have to install the necessary libraries one Library you have to install for reading the PDF file and then other libraries you have to install for the summarization using huging library in the Second Step you have to r

In [14]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap=50)

In [15]:
chunks = splitter.split_text(text)

In [16]:
len(chunks)

16

In [17]:
chunks

['hello everyone in my previous video I have already shown you how you can do text summarization using Haring library in this video I will show you how you can do text summarization from a PDF it means that you have a text in a PDF and you want to summarize the content using that PDF using Haring liary before starting let me give a short introduction about myself welcome to ad Academy the main Moto of this channel is AI for arm jenta my name is Dr Ayan deat I am an IIT Delhi alumni and FB research',
 "deat I am an IIT Delhi alumni and FB research scholar at Harvard University I have total 9 plus years of experience in the field of artificial intelligence deep learning machine learning NLP generative AI let's watch this video Welcome to My meta ver you can do teex separation in just three simple steps in the first step you have to install the necessary libraries one Library you have to install for reading the PDF file and then other libraries you have to install for the summarization us

In [20]:
embeddings = OpenAIEmbeddings(model='text-embedding-3-large', dimensions=50)

In [21]:
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x179e51ff0>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x128d2e1a0>, model='text-embedding-3-large', dimensions=50, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [26]:
vector_store = FAISS.from_texts(chunks,embeddings)

In [27]:
vector_store

In [28]:
vector_store.similarity_search(query="how to summarize text from pdf")

[Document(id='392e513d-bef1-48d7-8fea-d6a9bc3eaf02', metadata={}, page_content='hello everyone in my previous video I have already shown you how you can do text summarization using Haring library in this video I will show you how you can do text summarization from a PDF it means that you have a text in a PDF and you want to summarize the content using that PDF using Haring liary before starting let me give a short introduction about myself welcome to ad Academy the main Moto of this channel is AI for arm jenta my name is Dr Ayan deat I am an IIT Delhi alumni and FB research'),
 Document(id='4d5c2b33-de47-4c2e-bbe5-3755f0a45069', metadata={}, page_content="my summarized text okay here I have explained you these things now let me run it and execute and show it to you so you have to get a text first of all for that what I am doing you can get text from anywhere say I'm going to chat gbt and and I saying that tell me about suchin tul okay so chpt will give you some contents and just copy i

In [29]:
retriver = vector_store.as_retriever(search_kwargs={"k":4})

In [30]:
retriver.invoke("How are we summarising the pdf")

[Document(id='392e513d-bef1-48d7-8fea-d6a9bc3eaf02', metadata={}, page_content='hello everyone in my previous video I have already shown you how you can do text summarization using Haring library in this video I will show you how you can do text summarization from a PDF it means that you have a text in a PDF and you want to summarize the content using that PDF using Haring liary before starting let me give a short introduction about myself welcome to ad Academy the main Moto of this channel is AI for arm jenta my name is Dr Ayan deat I am an IIT Delhi alumni and FB research'),
 Document(id='4ae2628d-ae13-4036-bcc1-cae76a6bca37', metadata={}, page_content="task what task it will perform here I am doing salvation apart from the task you can give different things like which model you want to use different things you can give right now I am just giving what task I need only one parameter I I'm passing in P plan that is summarization and then it is creating a summarizer model which is an ob

In [34]:
prompt = PromptTemplate(template=""" You are a Helpful assistent. Answer from the following context.
               if context is insufficient then Directly say I dont know. 
               {context}
               Question {question}""",
               input_variable=['context','question'])

In [50]:
question = "what type of documents are talked in this video?"

In [51]:
context = retriver.invoke(question)

In [52]:
context = " ".join(i.page_content for i in context)

In [53]:
final_prompt = prompt.invoke({'question':question,'context':context})

In [54]:
llm = ChatOpenAI(model='gpt-3.5-turbo',temperature=0.2)

In [55]:
answer = llm.invoke(final_prompt)

In [56]:
answer.content

'The video talks about text summarization from a PDF document using the Haring library.'

In [22]:
vectors = embeddings.embed_documents(chunks)

In [23]:
vectors

[[-0.06658935546875,
  0.041412353515625,
  -0.042938232421875,
  -0.0654296875,
  0.0841064453125,
  -0.0921630859375,
  -0.10943603515625,
  0.29150390625,
  -0.254638671875,
  0.2098388671875,
  0.130859375,
  -0.06805419921875,
  0.1580810546875,
  -0.128662109375,
  -0.024383544921875,
  -0.130859375,
  -0.05426025390625,
  -0.09454345703125,
  0.046783447265625,
  -0.327392578125,
  0.1634521484375,
  -0.1549072265625,
  -0.255859375,
  -0.060943603515625,
  0.001316070556640625,
  0.066162109375,
  -0.1082763671875,
  -0.041717529296875,
  -0.073974609375,
  -0.01220703125,
  0.03924560546875,
  0.08837890625,
  -0.07611083984375,
  -0.260009765625,
  -0.1480712890625,
  -0.161376953125,
  0.1654052734375,
  -0.09576416015625,
  -0.03094482421875,
  0.11077880859375,
  0.228515625,
  0.2486572265625,
  -0.056549072265625,
  0.0290374755859375,
  -0.1036376953125,
  -0.166259765625,
  0.05511474609375,
  -0.114990234375,
  -0.1563720703125,
  -0.2066650390625],
 [0.0233154296875,

In [25]:
db = FAISS.from_documents(chunks,vectors)

AttributeError: 'str' object has no attribute 'page_content'